In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# open csvs
COLUMNS = ['k', 'precision@k', 'recall@k', 'maPrecision']

# the different tests that were run
TESTS = ['precision@k','recall@k','maPrecision']

DATASET = 'movielens'

# directory of the csv's
directory_list = os.listdir(f'{DATASET}/.') 

# Loop through each test type
for test in TESTS:

    # each dataset tested
    for test_results_file in directory_list:

        # check if folder
        if not 'describe' in test_results_file:

            # directory of dataset
            results_file_path = f'{DATASET}/{test_results_file}'          

            # read results
            algorithm_results = pd.read_csv(results_file_path, usecols= COLUMNS)

            # get x axis = k
            k = algorithm_results['k'].unique()

            # get averages for each
            algorithm_results = algorithm_results.groupby('k')[['precision@k', 'recall@k', 'maPrecision']].mean()

            # convert from decimal to percent
            algorithm_results = algorithm_results[['precision@k', 'recall@k', 'maPrecision']]

            # get test
            algorithm_test = algorithm_results[test].astype(float)

            A = test_results_file.split("A=")[1].split('_')[0]


            
            for i in range(100):     
                k = i+1       
                colour = (0,algorithm_test.loc[k],0)
                # Plotting both the curves simultaneously
                plt.scatter(A, k, color=colour, label=A, s=10)

        

    # Naming the x-axis, y-axis and the whole graph
    plt.title(f'{DATASET}\n{test}')
    plt.xlabel("A")
    plt.ylabel("K")

    # plt.title(test)

    # Adding legend, which helps us recognize the curve according to it's color
    # plt.legend()

    # save the figure
    plt.savefig(f'{DATASET}_{test}')

    plt.close()

In [6]:
import pandas as pd


FILE = "data_cleaned.csv"
COLUMN_NAMES = ["item_id","user_id","rating","ts"]
DELIMITER = ","
SKIPROWS = 0


df = pd.read_csv(FILE, delimiter=DELIMITER, encoding="utf-8", names=COLUMN_NAMES, skiprows=SKIPROWS)

df['ts'] = pd.to_datetime( df['ts'].astype(str), yearfirst=True )
df['ts'] = df['ts'].astype('int64') // 10**9

print(df)
print(f'users: {len(df['user_id'].unique())}')
print(f'items: {len(df['item_id'].unique())}')



FileNotFoundError: [Errno 2] No such file or directory: 'data_cleaned.csv'

In [2]:
''' Remove duplicate reviews. Keep most recent'''

df_grouped_by_user = df.groupby(['user_id', 'item_id']).count()['ts']
print(df_grouped_by_user[df_grouped_by_user > 1].describe())

df_sorted = df.sort_values(by=['ts'])
df_remove_duplicates = df_sorted.drop_duplicates(subset=['user_id','item_id'], keep='last')

print(f'users: {len(df_remove_duplicates['user_id'].unique())}')
print(f'items: {len(df_remove_duplicates['item_id'].unique())}')
print(df_remove_duplicates)

count    24053764.0
mean            2.0
std             0.0
min             2.0
25%             2.0
50%             2.0
75%             2.0
max             2.0
Name: ts, dtype: float64
users: 470758
items: 4499
          item_id  user_id  rating          ts
19585852     3730   510180       4   942278400
9056171      1798   510180       5   942278400
44447682     3870   510180       2   942278400
30955237     1367   510180       5   942278400
38946441     2866   510180       3   942278400
...           ...      ...     ...         ...
43994352     3817   350889       4  1135987200
18852137     3610  1343844       5  1135987200
15890079     3098  1110983       4  1135987200
31011701     1395  1066529       5  1135987200
1149487       260  1181079       4  1135987200

[24053764 rows x 4 columns]


In [3]:
''' Remove sparse items '''

df_grouped_by_item = df_remove_duplicates.groupby(['item_id']).count()['ts']
print(df_grouped_by_item.describe())
print(df_grouped_by_item)

df_no_sparse = df_remove_duplicates[df_remove_duplicates['item_id'].isin(df_grouped_by_item[df_grouped_by_item>=10].index)]
print(df_no_sparse)

count      4499.000000
mean       5346.468993
std       16176.313851
min          36.000000
25%         192.000000
50%         552.000000
75%        2538.000000
max      193941.000000
Name: ts, dtype: float64
item_id
1        547
2        145
3       2012
4        142
5       1140
        ... 
4495     614
4496    9519
4497     714
4498     269
4499     428
Name: ts, Length: 4499, dtype: int64
          item_id  user_id  rating          ts
19585852     3730   510180       4   942278400
9056171      1798   510180       5   942278400
44447682     3870   510180       2   942278400
30955237     1367   510180       5   942278400
38946441     2866   510180       3   942278400
...           ...      ...     ...         ...
43994352     3817   350889       4  1135987200
18852137     3610  1343844       5  1135987200
15890079     3098  1110983       4  1135987200
31011701     1395  1066529       5  1135987200
1149487       260  1181079       4  1135987200

[24053764 rows x 4 columns]


In [6]:
''' Select users randomly '''
unique_users = df_no_sparse['user_id'].unique()

import random

random.seed(45)
netflix_sample = df_no_sparse[df_no_sparse['user_id'].isin(random.choices( unique_users, k= 1000 ))]

netflix_sample

,item_id,user_id,rating,ts
11774433,2252,886919,4,947721600
19878472,3798,886919,3,947721600
46196419,4171,886919,2,947721600
46195862,4171,626377,3,947808000
14724613,2851,626377,3,947808000
...,...,...,...,...
9233455,1818,2099124,5,1135987200
40729992,3239,1414279,5,1135987200
44762184,3917,1721003,5,1135987200
44199317,3848,332152,4,1135987200


In [5]:
netflix_sample.to_csv('netflix_sample2.csv', index=False)